<a id="intro"></a>
# Лабораторная работа №8
## Исследование и прогнозирование исхода матчей Dota 2 на основе временных данных

### Цель работы

Цель работы — провести исследование датасета матчей Dota 2, выделить из него данные, подходящие для построения временного ряда, сформировать собственный аналитический датасет и решить задачу прогнозирования с использованием моделей машинного обучения и глубокого обучения.

В рамках работы рассматривается датасет **Dota 2 Matches**, содержащий несколько связанных таблиц с информацией о матчах, игроках, героях, предметах, событиях и поминутной статистике.

### Общий подход

На первом этапе выполняется обзор всех файлов датасета.  
Необходимо определить:

- какие таблицы входят в датасет;
- какие признаки доступны в каждой таблице;
- какие таблицы связаны между собой;
- какие данные можно использовать для построения временного ряда;
- какие признаки могут быть целевыми, а какие — экзогенными.

После изучения исходных таблиц будет сформирован итоговый датасет для моделирования.  
Только после этого выполняются EDA, генерация временных признаков и обучение моделей.

### Ограничение датасета

Датасет отражает состояние Dota 2 на момент сбора данных и не полностью соответствует текущей версии игры.

Это важно учитывать при интерпретации признаков:

- структура уровней героев в датасете ограничена значением `level = 25`;
- в современных версиях Dota 2 доступны более высокие уровни героя;
- в игре появились дополнительные механики, например таланты, аспекты, встроенные способности и изменения предметов;
- часть героев, предметов и способностей могла быть изменена, удалена или переработана.

Поэтому модель, построенная на этом датасете, рассматривается как исследовательская модель для исторических матчей из данного набора данных, а не как готовая модель для актуальной версии Dota 2.

<a id="imports"></a>

## 1. Импорты и настройки

В этом разделе подключаются необходимые библиотеки, фиксируется случайное зерно и задаются базовые настройки отображения таблиц и графиков.

In [5]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 140)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

CURRENT_DIR = Path.cwd()
DATA_DIR = CURRENT_DIR / "data"

print("Текущая рабочая директория:")
print(CURRENT_DIR)

print("\nПапка с данными:")
print(DATA_DIR)

files = sorted(DATA_DIR.iterdir())

print("\nКоличество файлов:", len(files))
print("\nФайлы датасета:")

for file in files:
    print(file.name)

Текущая рабочая директория:
c:\Users\motyn\Desktop\repositories\_OMSTU\Analyze\lab_7_8

Папка с данными:
c:\Users\motyn\Desktop\repositories\_OMSTU\Analyze\lab_7_8\data

Количество файлов: 19

Файлы датасета:
ability_ids.csv
ability_upgrades.csv
chat.csv
cluster_regions.csv
hero_names.csv
item_ids.csv
match.csv
match_outcomes.csv
objectives.csv
patch_dates.csv
player_ratings.csv
player_time.csv
players.csv
purchase_log.csv
teamfights.csv
teamfights_players.csv
test_labels.csv
test_player.csv
yasp_sample.json


<a id="dataset-files"></a>

## 2. Обзор файлов датасета

На данном этапе выполняется первичный обзор файлов датасета **Dota 2 Matches**.

Датасет содержит несколько CSV-таблиц и один JSON-файл. Перед построением итогового аналитического датасета необходимо понять, какие данные содержатся в каждом файле, какие таблицы являются основными, а какие выполняют роль справочников или дополнительных источников информации.

<a id="dataset-files-list"></a>

### 2.1. Список файлов датасета

Сначала выведем список файлов, которые входят в скачанный датасет.

In [6]:
dataset_files = []

for file_path in sorted(DATA_DIR.iterdir()):
    dataset_files.append({
        "file_name": file_path.name,
        "file_type": file_path.suffix.replace(".", ""),
        "size_mb": round(file_path.stat().st_size / 1024 / 1024, 2)
    })

dataset_files_df = pd.DataFrame(dataset_files)
dataset_files_df

,file_name,file_type,size_mb
0,ability_ids.csv,csv,0.02
1,ability_upgrades.csv,csv,178.33
2,chat.csv,csv,50.26
3,cluster_regions.csv,csv,0.00
4,hero_names.csv,csv,0.00
5,item_ids.csv,csv,0.00
6,match.csv,csv,2.57
7,match_outcomes.csv,csv,102.07
8,objectives.csv,csv,57.26
9,patch_dates.csv,csv,0.00


<a id="dataset-files-overview"></a>

### 2.2. Размеры и структура CSV-файлов

Для каждого CSV-файла посмотрим количество строк, количество столбцов и первые названия признаков.

Это нужно, чтобы предварительно разделить файлы на несколько групп:

- основные таблицы с матчами и игроками;
- таблицы с временными данными;
- таблицы с событиями матча;
- справочники;
- служебные таблицы.

In [7]:
csv_overview = []

for file_path in sorted(DATA_DIR.glob("*.csv")):
    df_head = pd.read_csv(file_path, nrows=5)
    rows_count = sum(1 for _ in open(file_path, encoding="utf-8")) - 1
    
    csv_overview.append({
        "file_name": file_path.name,
        "rows": rows_count,
        "columns": df_head.shape[1],
        "first_columns": ", ".join(df_head.columns[:10])
    })

csv_overview_df = pd.DataFrame(csv_overview)
csv_overview_df

,file_name,rows,columns,first_columns
0,ability_ids.csv,688,2,"ability_id, ability_name"
1,ability_upgrades.csv,8939599,5,"ability, level, time, player_slot, match_id"
2,chat.csv,1439488,5,"match_id, key, slot, time, unit"
3,cluster_regions.csv,53,2,"cluster, region"
4,hero_names.csv,112,3,"name, hero_id, localized_name"
5,item_ids.csv,189,2,"item_id, item_name"
6,match.csv,50000,13,"match_id, start_time, duration, tower_status_r..."
7,match_outcomes.csv,1828588,10,"match_id, account_id_0, account_id_1, account_..."
8,objectives.csv,1173396,9,"match_id, key, player1, player2, slot, subtype..."
9,patch_dates.csv,19,2,"patch_date, name"


<a id="tables-role"></a>

### 2.4. Предварительная классификация таблиц

После просмотра состава датасета можно разделить файлы на несколько групп.

**Основные таблицы:**

- `match.csv` — информация о матчах, включая длительность, сервер, игровые статусы зданий и целевой признак `radiant_win`;
- `player_time.csv` — поминутные значения `gold`, `xp` и `last hits` для каждого слота игрока;
- `players.csv` — итоговая статистика игроков за матч, герои, предметы и игровые действия.

**Событийные таблицы с временными метками:**

- `objectives.csv` — события, связанные с игровыми целями;
- `teamfights.csv` — информация о командных драках;
- `teamfights_players.csv` — вклад игроков в командные драки;
- `purchase_log.csv` — покупки предметов игроками;
- `ability_upgrades.csv` — прокачка способностей игроками.

Эти таблицы могут быть полезны для расширения временного ряда, так как содержат поле времени и позволяют агрегировать события по минутам матча.

**Справочники:**

- `hero_names.csv` — соответствие `hero_id` и названия героя;
- `item_ids.csv` — соответствие `item_id` и названия предмета;
- `ability_ids.csv` — соответствие `ability_id` и названия способности;
- `cluster_regions.csv` — соответствие кластера и региона;
- `patch_dates.csv` — даты игровых патчей.

**Служебные и дополнительные таблицы:**

- `player_ratings.csv` — рейтинг игроков;
- `match_outcomes.csv` — дополнительная информация об исходах матчей и составах игроков;
- `test_player.csv` и `test_labels.csv` — отдельная тестовая часть датасета;
- `yasp_sample.json` — пример структуры данных.

На данном этапе таблицы не исключаются окончательно. Сначала будет проведено исследование их структуры и связей, после чего будут выбраны источники признаков для итогового аналитического датасета.

<a id="tables-research"></a>

## 3. Первичное исследование таблиц

После общего обзора файлов перейдём к изучению содержимого основных таблиц датасета.  
На этом этапе рассматриваются названия признаков и первые строки таблиц, чтобы определить их роль в дальнейшем построении временного ряда.

In [11]:
main_tables = ["match.csv", "players.csv", "player_time.csv"]

for table_name in main_tables:
    table_path = DATA_DIR / table_name
    df = pd.read_csv(table_path, nrows=5)
    
    print(f"Таблица: {table_name}")
    print("Размер примера:", df.shape)
    print("Колонки:")
    print(list(df.columns))
    display(df)

Таблица: match.csv
Размер примера: (5, 13)
Колонки:
['match_id', 'start_time', 'duration', 'tower_status_radiant', 'tower_status_dire', 'barracks_status_dire', 'barracks_status_radiant', 'first_blood_time', 'game_mode', 'radiant_win', 'negative_votes', 'positive_votes', 'cluster']


,match_id,start_time,duration,tower_status_radiant,tower_status_dire,barracks_status_dire,barracks_status_radiant,first_blood_time,game_mode,radiant_win,negative_votes,positive_votes,cluster
0,0,1446750112,2375,1982,4,3,63,1,22,True,0,1,155
1,1,1446753078,2582,0,1846,63,0,221,22,False,0,2,154
2,2,1446764586,2716,256,1972,63,48,190,22,False,0,0,132
3,3,1446765723,3085,4,1924,51,3,40,22,False,0,0,191
4,4,1446796385,1887,2047,0,0,63,58,22,True,0,0,156


Таблица: players.csv
Размер примера: (5, 73)
Колонки:
['match_id', 'account_id', 'hero_id', 'player_slot', 'gold', 'gold_spent', 'gold_per_min', 'xp_per_min', 'kills', 'deaths', 'assists', 'denies', 'last_hits', 'stuns', 'hero_damage', 'hero_healing', 'tower_damage', 'item_0', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5', 'level', 'leaver_status', 'xp_hero', 'xp_creep', 'xp_roshan', 'xp_other', 'gold_other', 'gold_death', 'gold_buyback', 'gold_abandon', 'gold_sell', 'gold_destroying_structure', 'gold_killing_heros', 'gold_killing_creeps', 'gold_killing_roshan', 'gold_killing_couriers', 'unit_order_none', 'unit_order_move_to_position', 'unit_order_move_to_target', 'unit_order_attack_move', 'unit_order_attack_target', 'unit_order_cast_position', 'unit_order_cast_target', 'unit_order_cast_target_tree', 'unit_order_cast_no_target', 'unit_order_cast_toggle', 'unit_order_hold_position', 'unit_order_train_ability', 'unit_order_drop_item', 'unit_order_give_item', 'unit_order_pickup_item', 

,match_id,account_id,hero_id,player_slot,gold,gold_spent,gold_per_min,xp_per_min,kills,deaths,assists,denies,last_hits,stuns,hero_damage,hero_healing,tower_damage,item_0,item_1,item_2,item_3,item_4,item_5,level,leaver_status,xp_hero,xp_creep,xp_roshan,xp_other,gold_other,gold_death,gold_buyback,gold_abandon,gold_sell,gold_destroying_structure,gold_killing_heros,gold_killing_creeps,gold_killing_roshan,gold_killing_couriers,unit_order_none,unit_order_move_to_position,unit_order_move_to_target,unit_order_attack_move,unit_order_attack_target,unit_order_cast_position,unit_order_cast_target,unit_order_cast_target_tree,unit_order_cast_no_target,unit_order_cast_toggle,unit_order_hold_position,unit_order_train_ability,unit_order_drop_item,unit_order_give_item,unit_order_pickup_item,unit_order_pickup_rune,unit_order_purchase_item,unit_order_sell_item,unit_order_disassemble_item,unit_order_move_item,unit_order_cast_toggle_auto,unit_order_stop,unit_order_taunt,unit_order_buyback,unit_order_glyph,unit_order_eject_item_from_stash,unit_order_cast_rune,unit_order_ping_ability,unit_order_move_to_direction,unit_order_patrol,unit_order_vector_target_position,unit_order_radar,unit_order_set_item_combine_lock,unit_order_continue
0,0,0,86,0,3261,10960,347,362,9,3,18,1,30,76.7356,8690,218,143,180,37,73,56,108,0,16,0,8840.0,5440.0,NaN,83.0,50.0,-957.0,NaN,NaN,212.0,3120.0,5145.0,1087.0,400.0,NaN,NaN,4070.0,1.0,25.0,416.0,51.0,144.0,3.0,71.0,NaN,188.0,16.0,NaN,NaN,NaN,2.0,35.0,2.0,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1,51,1,2954,17760,494,659,13,3,18,9,109,87.4164,23747,0,423,46,63,119,102,24,108,22,0,14331.0,8440.0,2683.0,671.0,395.0,-1137.0,NaN,NaN,1650.0,3299.0,6676.0,4317.0,937.0,NaN,NaN,5894.0,214.0,165.0,1031.0,98.0,39.0,4.0,439.0,NaN,346.0,22.0,NaN,NaN,12.0,52.0,30.0,4.0,NaN,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN
2,0,0,83,2,110,12195,350,385,0,4,15,1,58,NaN,4217,1595,399,48,60,59,108,65,0,17,0,6692.0,8112.0,NaN,453.0,259.0,-1436.0,-1015.0,NaN,NaN,3142.0,2418.0,3697.0,400.0,NaN,NaN,7053.0,3.0,132.0,645.0,36.0,160.0,20.0,373.0,NaN,643.0,17.0,5.0,NaN,7.0,8.0,28.0,NaN,1.0,18.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN
3,0,2,11,3,1179,22505,599,605,8,4,19,6,271,NaN,14832,2714,6055,63,147,154,164,79,160,21,0,8583.0,14230.0,894.0,293.0,100.0,-2156.0,NaN,NaN,938.0,4714.0,4104.0,10432.0,400.0,NaN,NaN,4712.0,133.0,163.0,690.0,9.0,15.0,7.0,406.0,NaN,150.0,21.0,NaN,NaN,1.0,9.0,45.0,7.0,NaN,14.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN
4,0,3,67,4,3307,23825,613,762,20,3,17,13,245,NaN,33740,243,1833,114,92,147,0,137,63,24,0,15814.0,14325.0,NaN,62.0,NaN,-1437.0,-1056.0,NaN,4194.0,3217.0,7467.0,9220.0,400.0,NaN,NaN,3853.0,7.0,7.0,1173.0,31.0,84.0,8.0,198.0,NaN,111.0,23.0,1.0,NaN,NaN,2.0,44.0,6.0,NaN,13.0,NaN,NaN,NaN,1.0,3.0,NaN,NaN,23.0,NaN,NaN,NaN,NaN,NaN,NaN


Таблица: player_time.csv
Размер примера: (5, 32)
Колонки:
['match_id', 'times', 'gold_t_0', 'lh_t_0', 'xp_t_0', 'gold_t_1', 'lh_t_1', 'xp_t_1', 'gold_t_2', 'lh_t_2', 'xp_t_2', 'gold_t_3', 'lh_t_3', 'xp_t_3', 'gold_t_4', 'lh_t_4', 'xp_t_4', 'gold_t_128', 'lh_t_128', 'xp_t_128', 'gold_t_129', 'lh_t_129', 'xp_t_129', 'gold_t_130', 'lh_t_130', 'xp_t_130', 'gold_t_131', 'lh_t_131', 'xp_t_131', 'gold_t_132', 'lh_t_132', 'xp_t_132']


,match_id,times,gold_t_0,lh_t_0,xp_t_0,gold_t_1,lh_t_1,xp_t_1,gold_t_2,lh_t_2,xp_t_2,gold_t_3,lh_t_3,xp_t_3,gold_t_4,lh_t_4,xp_t_4,gold_t_128,lh_t_128,xp_t_128,gold_t_129,lh_t_129,xp_t_129,gold_t_130,lh_t_130,xp_t_130,gold_t_131,lh_t_131,xp_t_131,gold_t_132,lh_t_132,xp_t_132
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,60,409,0,63,142,1,186,168,0,125,200,0,193,194,1,125,174,2,77,138,1,62,345,6,351,100,0,77,613,1,125
2,0,120,546,0,283,622,4,645,330,0,376,345,1,698,628,5,374,354,4,437,673,5,543,684,12,805,200,0,210,815,5,323
3,0,180,683,1,314,927,9,1202,430,0,376,644,6,1172,806,7,570,614,8,829,895,8,842,958,16,1135,300,0,210,1290,8,527
4,0,240,956,1,485,1264,11,1583,530,0,391,919,11,1610,1281,10,1216,1082,8,1318,1087,10,1048,1500,26,1842,400,0,210,1431,9,589


На основе первых строк основных таблиц можно сделать следующие выводы:

- `match.csv` содержит одну строку на матч и включает целевой признак `radiant_win`;
- `players.csv` содержит одну строку на игрока в матче, то есть для каждого матча обычно представлено 10 строк;
- `player_time.csv` содержит поминутные значения `gold`, `xp` и `last hits` по слотам игроков;
- признаки в `player_time.csv` разделены по слотам игроков: `0–4` относятся к Radiant, `128–132` относятся к Dire.

Таким образом, именно `player_time.csv` является главным источником временной структуры, а `match.csv` нужен для добавления целевой переменной.

<a id="event-tables"></a>

### 3.1. Исследование событийных таблиц

Кроме основных таблиц, в датасете есть несколько событийных таблиц.  
Они содержат временные метки и потенциально могут использоваться для построения дополнительных признаков на каждом временном срезе матча.

В этом разделе рассмотрим следующие таблицы:

- `objectives.csv` — игровые цели и события на карте;
- `teamfights.csv` — командные драки;
- `teamfights_players.csv` — статистика игроков в командных драках;
- `purchase_log.csv` — покупки предметов;
- `ability_upgrades.csv` — прокачка способностей.

Главный критерий полезности таблицы для нашей задачи — наличие `match_id` и временного признака, по которому данные можно агрегировать к минутам матча.

In [14]:
event_tables = [
    "objectives.csv",
    "teamfights.csv",
    "teamfights_players.csv",
    "purchase_log.csv",
    "ability_upgrades.csv"
]

for table_name in event_tables:
    table_path = DATA_DIR / table_name
    df = pd.read_csv(table_path, nrows=5)
    
    print(f"Таблица: {table_name}")
    print("Размер примера:", df.shape)
    print("Колонки:")
    print(list(df.columns))
    display(df)

Таблица: objectives.csv
Размер примера: (5, 9)
Колонки:
['match_id', 'key', 'player1', 'player2', 'slot', 'subtype', 'team', 'time', 'value']


,match_id,key,player1,player2,slot,subtype,team,time,value
0,0,NaN,0,6,0.0,CHAT_MESSAGE_FIRSTBLOOD,NaN,1,309
1,0,NaN,3,-1,3.0,CHAT_MESSAGE_TOWER_KILL,2.0,894,2
2,0,NaN,2,-1,NaN,CHAT_MESSAGE_ROSHAN_KILL,2.0,925,200
3,0,NaN,1,-1,1.0,CHAT_MESSAGE_AEGIS,NaN,925,0
4,0,NaN,7,-1,7.0,CHAT_MESSAGE_TOWER_KILL,3.0,1016,3


Таблица: teamfights.csv
Размер примера: (5, 5)
Колонки:
['match_id', 'start', 'end', 'last_death', 'deaths']


,match_id,start,end,last_death,deaths
0,0,220,252,237,3
1,0,429,475,460,3
2,0,900,936,921,3
3,0,1284,1328,1313,3
4,0,1614,1666,1651,5


Таблица: teamfights_players.csv
Размер примера: (5, 8)
Колонки:
['match_id', 'player_slot', 'buybacks', 'damage', 'deaths', 'gold_delta', 'xp_end', 'xp_start']


,match_id,player_slot,buybacks,damage,deaths,gold_delta,xp_end,xp_start
0,0,0,0,105,0,173,536,314
1,0,1,0,566,1,0,1583,1418
2,0,2,0,0,0,0,391,391
3,0,3,0,0,0,123,1775,1419
4,0,4,0,444,0,336,1267,983


Таблица: purchase_log.csv
Размер примера: (5, 4)
Колонки:
['item_id', 'time', 'player_slot', 'match_id']


,item_id,time,player_slot,match_id
0,44,-81,0,0
1,29,-63,0,0
2,43,6,0,0
3,84,182,0,0
4,46,197,0,0


Таблица: ability_upgrades.csv
Размер примера: (5, 5)
Колонки:
['ability', 'level', 'time', 'player_slot', 'match_id']


,ability,level,time,player_slot,match_id
0,5448,1,326,0,0
1,5450,2,452,0,0
2,5450,3,582,0,0
3,5448,4,804,0,0
4,5450,5,916,0,0


По результатам первичного просмотра видно, что событийные таблицы содержат разные типы информации.

`objectives.csv`, `purchase_log.csv` и `ability_upgrades.csv` имеют явное поле `time`, поэтому их можно агрегировать к минутам матча напрямую.

`teamfights.csv` содержит временные границы командных драк: `start`, `end`, `last_death`.

`teamfights_players.csv` содержит статистику игроков в драках, но не имеет отдельного поля времени. Для использования этой таблицы её необходимо связывать с `teamfights.csv`. На базовом этапе можно использовать только агрегаты из `teamfights.csv`, а расширение через `teamfights_players.csv` рассматривать как дополнительный источник признаков.

Пропуски в событийных таблицах не всегда являются ошибками данных. Часть полей заполняется только для определённых типов событий.

In [15]:
event_missing_overview = []

for table_name in event_tables:
    table_path = DATA_DIR / table_name
    df = pd.read_csv(table_path)
    
    missing_percent = df.isna().mean().sort_values(ascending=False) * 100
    
    event_missing_overview.append({
        "file_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "columns_with_missing": int((df.isna().sum() > 0).sum()),
        "top_missing_columns": ", ".join(
            missing_percent[missing_percent > 0].head(5).round(2).astype(str).index
        )
    })

event_missing_overview_df = pd.DataFrame(event_missing_overview)
event_missing_overview_df

,file_name,rows,columns,columns_with_missing,top_missing_columns
0,objectives.csv,1173396,9,3,"key, team, slot"
1,teamfights.csv,539047,5,0,
2,teamfights_players.csv,5390470,8,0,
3,purchase_log.csv,18193745,4,0,
4,ability_upgrades.csv,8939599,5,0,


<a id="objectives-table"></a>

### 3.2. Исследование таблицы `objectives.csv`

Таблица `objectives.csv` содержит события, связанные с игровыми целями и важными моментами матча.

В таблице есть временная метка `time`, поэтому события можно анализировать относительно времени матча и потенциально агрегировать к временным срезам.

На данном этапе необходимо изучить:

- какие типы событий содержатся в поле `subtype`;
- насколько часто встречается каждый тип события;
- какие поля заполнены или пропущены для разных типов событий.

In [16]:
objectives = pd.read_csv(DATA_DIR / "objectives.csv")

print("Размер objectives.csv:", objectives.shape)
print("\nТипы данных:")
display(objectives.dtypes)

print("\nПервые строки:")
display(objectives.head(10))

print("\nКоличество пропусков:")
display(objectives.isna().sum())

print("\nДоля пропусков, %:")
display((objectives.isna().mean() * 100).round(2))

Размер objectives.csv: (1173396, 9)

Типы данных:


match_id      int64
key         float64
player1       int64
player2       int64
slot        float64
subtype         str
team        float64
time          int64
value         int64
dtype: object


Первые строки:


,match_id,key,player1,player2,slot,subtype,team,time,value
0,0,NaN,0,6,0.0,CHAT_MESSAGE_FIRSTBLOOD,NaN,1,309
1,0,NaN,3,-1,3.0,CHAT_MESSAGE_TOWER_KILL,2.0,894,2
2,0,NaN,2,-1,NaN,CHAT_MESSAGE_ROSHAN_KILL,2.0,925,200
3,0,NaN,1,-1,1.0,CHAT_MESSAGE_AEGIS,NaN,925,0
4,0,NaN,7,-1,7.0,CHAT_MESSAGE_TOWER_KILL,3.0,1016,3
5,0,NaN,3,-1,3.0,CHAT_MESSAGE_TOWER_KILL,2.0,1024,2
6,0,NaN,-1,-1,-1.0,CHAT_MESSAGE_TOWER_KILL,2.0,1446,2
7,0,NaN,4,-1,4.0,CHAT_MESSAGE_TOWER_KILL,2.0,1590,2
8,0,NaN,2,-1,NaN,CHAT_MESSAGE_ROSHAN_KILL,2.0,1740,200
9,0,NaN,1,-1,1.0,CHAT_MESSAGE_AEGIS,NaN,1740,0



Количество пропусков:


match_id         0
key         903820
player1          0
player2          0
slot        346543
subtype          0
team        394641
time             0
value            0
dtype: int64


Доля пропусков, %:


match_id     0.00
key         77.03
player1      0.00
player2      0.00
slot        29.53
subtype      0.00
team        33.63
time         0.00
value        0.00
dtype: float64

In [17]:
objectives_subtypes = (
    objectives["subtype"]
    .value_counts()
    .reset_index()
)

objectives_subtypes.columns = ["subtype", "count"]
objectives_subtypes["share_percent"] = (
    objectives_subtypes["count"] / len(objectives) * 100
).round(2)

objectives_subtypes

,subtype,count,share_percent
0,CHAT_MESSAGE_TOWER_KILL,663032,56.51
1,CHAT_MESSAGE_BARRACKS_KILL,269576,22.97
2,CHAT_MESSAGE_ROSHAN_KILL,76967,6.56
3,CHAT_MESSAGE_AEGIS,75174,6.41
4,CHAT_MESSAGE_FIRSTBLOOD,48721,4.15
5,CHAT_MESSAGE_TOWER_DENY,38756,3.30
6,CHAT_MESSAGE_AEGIS_STOLEN,1170,0.10


In [18]:
objectives_missing_by_subtype = (
    objectives
    .groupby("subtype")
    [["key", "slot", "team"]]
    .apply(lambda part: part.isna().mean() * 100)
    .round(2)
    .reset_index()
)

objectives_missing_by_subtype

,subtype,key,slot,team
0,CHAT_MESSAGE_AEGIS,100.0,0.0,100.0
1,CHAT_MESSAGE_AEGIS_STOLEN,100.0,0.0,100.0
2,CHAT_MESSAGE_BARRACKS_KILL,0.0,100.0,100.0
3,CHAT_MESSAGE_FIRSTBLOOD,100.0,0.0,100.0
4,CHAT_MESSAGE_ROSHAN_KILL,100.0,100.0,0.0
5,CHAT_MESSAGE_TOWER_DENY,100.0,0.0,0.0
6,CHAT_MESSAGE_TOWER_KILL,100.0,0.0,0.0


Наиболее частыми событиями в `objectives.csv` являются уничтожение башен и бараков.  
Эти события напрямую связаны с преимуществом команды по карте и могут быть полезны для прогнозирования исхода матча.

Пропуски в таблице зависят от типа события:

- для `CHAT_MESSAGE_TOWER_KILL` и `CHAT_MESSAGE_TOWER_DENY` заполнены `slot` и `team`, но не заполнен `key`;
- для `CHAT_MESSAGE_BARRACKS_KILL` заполнен `key`, но не заполнены `slot` и `team`;
- для `CHAT_MESSAGE_ROSHAN_KILL` заполнен `team`, но не заполнены `key` и `slot`;
- для `CHAT_MESSAGE_AEGIS`, `CHAT_MESSAGE_AEGIS_STOLEN` и `CHAT_MESSAGE_FIRSTBLOOD` заполнен `slot`, но не заполнены `key` и `team`.

Следовательно, пропуски в таблице являются структурными: разные типы событий используют разные поля.

In [22]:
for subtype in objectives["subtype"].unique():
    part = objectives[objectives["subtype"] == subtype]
    
    print("=" * 100)
    print(subtype)
    print("Количество событий:", len(part))
    
    print("\nУникальные значения team:")
    print(sorted(part["team"].dropna().unique()))
    
    print("\nУникальные значения slot:")
    print(sorted(part["slot"].dropna().unique()))
    
    print("\nУникальные значения value:")
    print(sorted(part["value"].dropna().unique())[:30])
    print("\n")

CHAT_MESSAGE_FIRSTBLOOD
Количество событий: 48721

Уникальные значения team:
[]

Уникальные значения slot:
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0)]

Уникальные значения value:
[np.int64(269), np.int64(279), np.int64(283), np.int64(284), np.int64(285), np.int64(286), np.int64(287), np.int64(288), np.int64(289), np.int64(290), np.int64(291), np.int64(292), np.int64(293), np.int64(294), np.int64(295), np.int64(296), np.int64(297), np.int64(298), np.int64(299), np.int64(300), np.int64(301), np.int64(302), np.int64(303), np.int64(304), np.int64(305), np.int64(306), np.int64(307), np.int64(308), np.int64(309), np.int64(310)]


CHAT_MESSAGE_TOWER_KILL
Количество событий: 663032

Уникальные значения team:
[np.float64(2.0), np.float64(3.0)]

Уникальные значения slot:
[np.float64(-1.0), np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0

<a id="objectives-interpretation"></a>

### 3.2.1. Интерпретация полей таблицы `objectives.csv`

Таблица `objectives.csv` содержит события, связанные с важными целями на карте.

Основные поля таблицы:

- `match_id` — идентификатор матча;
- `subtype` — тип события;
- `time` — время события в секундах от начала матча;
- `slot` — слот игрока, если событие связано с конкретным игроком;
- `team` — команда или сторона, связанная с событием;
- `value` — дополнительное значение события, смысл которого зависит от `subtype`;
- `key` — дополнительный код события, используемый не для всех типов событий;
- `player1`, `player2` — дополнительные поля, связанные с участниками события.

По результатам анализа видно, что `subtype` определяет, какие поля имеют смысл:

| subtype | Интерпретация |
|---|---|
| `CHAT_MESSAGE_FIRSTBLOOD` | событие первой крови |
| `CHAT_MESSAGE_TOWER_KILL` | уничтожение башни |
| `CHAT_MESSAGE_TOWER_DENY` | deny башни |
| `CHAT_MESSAGE_BARRACKS_KILL` | уничтожение бараков |
| `CHAT_MESSAGE_ROSHAN_KILL` | убийство Рошана |
| `CHAT_MESSAGE_AEGIS` | получение Aegis |
| `CHAT_MESSAGE_AEGIS_STOLEN` | кража Aegis |

Для событий, связанных с игроком, используется поле `slot`.  
В этой таблице слоты представлены значениями от `0` до `9`:

- `0–4` — игроки Radiant;
- `5–9` — игроки Dire.

Для событий, связанных с командой, используется поле `team`.  
В событиях `TOWER_KILL` и `ROSHAN_KILL` встречаются значения `2` и `3`, которые требуют аккуратной интерпретации при построении признаков.

На данном этапе важно не только сохранить события, но и понять, как их можно преобразовать в признаки временного ряда. Например:

- количество уничтоженных башен к текущей минуте;
- количество убитых Рошанов к текущей минуте;
- факт первой крови;
- команда, получившая Aegis;
- количество уничтоженных бараков.

Так как некоторые поля имеют разный смысл для разных `subtype`, дальнейшая обработка `objectives.csv` должна выполняться отдельно по типам событий.

In [23]:
objectives_interpretation = objectives.copy()

objectives_interpretation["minute"] = (objectives_interpretation["time"] // 60).astype(int)

objectives_interpretation["slot_team"] = np.where(
    objectives_interpretation["slot"].between(0, 4),
    "Radiant",
    np.where(
        objectives_interpretation["slot"].between(5, 9),
        "Dire",
        "No player slot"
    )
)

objectives_interpretation_summary = (
    objectives_interpretation
    .groupby(["subtype", "slot_team"])
    .size()
    .reset_index(name="events_count")
    .sort_values(["subtype", "events_count"], ascending=[True, False])
)

objectives_interpretation_summary

,subtype,slot_team,events_count
0,CHAT_MESSAGE_AEGIS,Dire,45016
1,CHAT_MESSAGE_AEGIS,Radiant,30158
2,CHAT_MESSAGE_AEGIS_STOLEN,Dire,633
3,CHAT_MESSAGE_AEGIS_STOLEN,Radiant,537
4,CHAT_MESSAGE_BARRACKS_KILL,No player slot,269576
5,CHAT_MESSAGE_FIRSTBLOOD,Dire,24758
6,CHAT_MESSAGE_FIRSTBLOOD,Radiant,23963
7,CHAT_MESSAGE_ROSHAN_KILL,No player slot,76967
9,CHAT_MESSAGE_TOWER_DENY,Radiant,19417
8,CHAT_MESSAGE_TOWER_DENY,Dire,19339


In [24]:
team_based_objectives = objectives[
    objectives["subtype"].isin([
        "CHAT_MESSAGE_TOWER_KILL",
        "CHAT_MESSAGE_TOWER_DENY",
        "CHAT_MESSAGE_ROSHAN_KILL"
    ])
].copy()

team_value_summary = (
    team_based_objectives
    .groupby(["subtype", "team", "value"])
    .size()
    .reset_index(name="events_count")
    .sort_values(["subtype", "team", "value"])
)

team_value_summary

,subtype,team,value,events_count
0,CHAT_MESSAGE_ROSHAN_KILL,2.0,200,31084
1,CHAT_MESSAGE_ROSHAN_KILL,3.0,200,45883
2,CHAT_MESSAGE_TOWER_DENY,80.0,80,29267
3,CHAT_MESSAGE_TOWER_DENY,100.0,100,8090
4,CHAT_MESSAGE_TOWER_DENY,120.0,120,1062
5,CHAT_MESSAGE_TOWER_DENY,140.0,140,337
6,CHAT_MESSAGE_TOWER_KILL,2.0,2,340818
7,CHAT_MESSAGE_TOWER_KILL,3.0,3,322214


In [25]:
barracks_events = objectives[
    objectives["subtype"] == "CHAT_MESSAGE_BARRACKS_KILL"
].copy()

barracks_summary = (
    barracks_events
    .groupby(["key", "value"])
    .size()
    .reset_index(name="events_count")
    .sort_values(["key", "value"])
)

barracks_summary.head(50)

,key,value,events_count
0,1.0,1,21292
1,2.0,2,21680
2,4.0,4,27570
3,8.0,8,27636
4,16.0,16,19149
5,32.0,32,19289
6,64.0,64,21736
7,128.0,128,21811
8,256.0,256,25742
9,512.0,512,25708


In [26]:
match_targets = pd.read_csv(
    DATA_DIR / "match.csv",
    usecols=["match_id", "radiant_win"]
)

objectives_with_target = objectives.merge(
    match_targets,
    on="match_id",
    how="left"
)

objective_win_summary = (
    objectives_with_target
    .groupby(["subtype", "team"])["radiant_win"]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={
        "count": "events_count",
        "mean": "radiant_win_rate"
    })
    .sort_values(["subtype", "team"])
)

objective_win_summary

,subtype,team,events_count,radiant_win_rate
0,CHAT_MESSAGE_ROSHAN_KILL,2.0,31084,0.785324
1,CHAT_MESSAGE_ROSHAN_KILL,3.0,45883,0.283482
2,CHAT_MESSAGE_TOWER_DENY,80.0,29267,0.504937
3,CHAT_MESSAGE_TOWER_DENY,100.0,8090,0.485167
4,CHAT_MESSAGE_TOWER_DENY,120.0,1062,0.509416
5,CHAT_MESSAGE_TOWER_DENY,140.0,337,0.531157
6,CHAT_MESSAGE_TOWER_KILL,2.0,340818,0.756662
7,CHAT_MESSAGE_TOWER_KILL,3.0,322214,0.249902


По результатам проверки `team` для событий `TOWER_KILL` и `ROSHAN_KILL` можно принять следующую интерпретацию:

- `team = 2` — событие в пользу Radiant;
- `team = 3` — событие в пользу Dire.

Это подтверждается долей побед Radiant: для событий с `team = 2` она значительно выше, а для событий с `team = 3` значительно ниже.

Для `TOWER_DENY` поле `team` имеет значения `80`, `100`, `120`, `140`, поэтому его нельзя использовать как код стороны. Такие события лучше обрабатывать через `slot`, где `0–4` соответствуют Radiant, а `5–9` — Dire.

Для `BARRACKS_KILL` поля `key` и `value` кодируют тип разрушенного барака, поэтому эту таблицу стоит использовать как счётчик уничтоженных бараков без детальной расшифровки конкретного барака на текущем этапе.

<a id="teamfights-table"></a>

### 3.3. Исследование таблицы `teamfights.csv`

Таблица `teamfights.csv` содержит информацию о командных драках.

Для каждой драки указаны:

- `match_id` — идентификатор матча;
- `start` — время начала драки;
- `end` — время окончания драки;
- `last_death` — время последней смерти в драке;
- `deaths` — количество смертей в драке.

Эта таблица может быть использована для создания временных признаков, например количества драк и смертей к текущей минуте матча.

In [27]:
teamfights = pd.read_csv(DATA_DIR / "teamfights.csv")

print("Размер teamfights.csv:", teamfights.shape)

print("\nТипы данных:")
display(teamfights.dtypes)

print("\nПервые строки:")
display(teamfights.head(10))

print("\nПропуски:")
display(teamfights.isna().sum())

print("\nОписательная статистика:")
display(teamfights[["start", "end", "last_death", "deaths"]].describe())

Размер teamfights.csv: (539047, 5)

Типы данных:


match_id      int64
start         int64
end           int64
last_death    int64
deaths        int64
dtype: object


Первые строки:


,match_id,start,end,last_death,deaths
0,0,220,252,237,3
1,0,429,475,460,3
2,0,900,936,921,3
3,0,1284,1328,1313,3
4,0,1614,1666,1651,5
5,0,1672,1709,1694,3
6,0,1734,1783,1768,6
7,0,1818,1867,1852,5
8,0,1863,1912,1897,5
9,0,2101,2145,2130,4



Пропуски:


match_id      0
start         0
end           0
last_death    0
deaths        0
dtype: int64


Описательная статистика:


,start,end,last_death,deaths
count,539047.000000,539047.000000,539047.000000,539047.000000
mean,1524.624868,1570.802991,1555.802991,4.324430
std,779.104437,779.974151,779.974151,1.522701
min,-49.000000,-1.000000,-16.000000,3.000000
25%,923.000000,968.000000,953.000000,3.000000
50%,1464.000000,1510.000000,1495.000000,4.000000
75%,2039.000000,2086.000000,2071.000000,5.000000
max,15941.000000,15979.000000,15964.000000,23.000000


В таблице `teamfights.csv` пропусков нет, все признаки числовые.

Минимальные значения `start`, `end` и `last_death` меньше нуля, значит часть событий относится к периоду до формального начала матча. При построении временных признаков такие значения нужно обработать отдельно.

Для дальнейшего моделирования таблица может дать признаки:

- количество командных драк к текущей минуте;
- количество смертей в командных драках к текущей минуте;
- количество командных драк за последние несколько минут.

<a id="teamfights-players-table"></a>

### 3.4. Исследование таблицы `teamfights_players.csv`

Таблица `teamfights_players.csv` содержит статистику игроков внутри командных драк.

В таблице нет собственного поля времени, поэтому её нельзя напрямую агрегировать по минутам. Для использования этих данных нужно связать строки с таблицей `teamfights.csv`, где указаны временные границы драки.

In [28]:
teamfights_players = pd.read_csv(DATA_DIR / "teamfights_players.csv")

print("Размер teamfights_players.csv:", teamfights_players.shape)

print("\nТипы данных:")
display(teamfights_players.dtypes)

print("\nПервые строки:")
display(teamfights_players.head(10))

print("\nПропуски:")
display(teamfights_players.isna().sum())

print("\nОписательная статистика:")
display(teamfights_players[["buybacks", "damage", "deaths", "gold_delta", "xp_start", "xp_end"]].describe())

Размер teamfights_players.csv: (5390470, 8)

Типы данных:


match_id       int64
player_slot    int64
buybacks       int64
damage         int64
deaths         int64
gold_delta     int64
xp_end         int64
xp_start       int64
dtype: object


Первые строки:


,match_id,player_slot,buybacks,damage,deaths,gold_delta,xp_end,xp_start
0,0,0,0,105,0,173,536,314
1,0,1,0,566,1,0,1583,1418
2,0,2,0,0,0,0,391,391
3,0,3,0,0,0,123,1775,1419
4,0,4,0,444,0,336,1267,983
5,0,128,0,477,1,249,1318,1035
6,0,129,0,636,1,-27,1048,904
7,0,130,0,0,0,190,1904,1589
8,0,131,0,0,0,0,210,210
9,0,132,0,0,0,378,659,589



Пропуски:


match_id       0
player_slot    0
buybacks       0
damage         0
deaths         0
gold_delta     0
xp_end         0
xp_start       0
dtype: int64


Описательная статистика:


,buybacks,damage,deaths,gold_delta,xp_start,xp_end
count,5.390470e+06,5.390470e+06,5.390470e+06,5.390470e+06,5.390470e+06,5.390470e+06
mean,2.276147e-02,1.024086e+03,4.374474e-01,2.580012e+02,1.006097e+04,1.071467e+04
std,1.491422e-01,1.423389e+03,5.457010e-01,5.787074e+02,7.491389e+03,7.742108e+03
min,0.000000e+00,0.000000e+00,0.000000e+00,-5.562000e+03,0.000000e+00,0.000000e+00
25%,0.000000e+00,1.480000e+02,0.000000e+00,-8.900000e+01,4.282000e+03,4.696000e+03
50%,0.000000e+00,6.040000e+02,0.000000e+00,1.660000e+02,8.207000e+03,8.819000e+03
75%,0.000000e+00,1.353000e+03,1.000000e+00,5.400000e+02,1.399200e+04,1.488600e+04
max,1.000000e+00,8.001000e+04,1.500000e+01,4.880000e+04,3.457400e+04,3.457400e+04


Таблица `teamfights_players.csv` содержит статистику игроков внутри командных драк.  
Каждая командная драка представлена 10 строками — по одной строке на каждого игрока.

Поле `player_slot` позволяет определить сторону игрока:

- `0–4` — Radiant;
- `128–132` — Dire.

Интерпретация основных признаков:

- `damage` — урон, нанесённый игроком в драке;
- `deaths` — погиб ли игрок в драке;
- `buybacks` — использовал ли игрок buyback;
- `gold_delta` — изменение золота игрока по итогам драки;
- `xp_start` и `xp_end` — значения опыта до и после драки.

Отрицательные значения `gold_delta` не являются ошибкой: они могут означать, что игрок потерял золото в результате смерти или других игровых событий, связанных с дракой. Положительные значения, наоборот, показывают экономическую выгоду игрока.

Для модели эта таблица потенциально полезна, потому что позволяет оценивать не только факт драки, но и её результат:

- какая команда нанесла больше урона;
- какая команда потеряла больше героев;
- какая команда получила больше золота;
- какая команда получила больше опыта;
- были ли использованы buyback.

Так как в таблице нет отдельного поля времени, для построения временных признаков её нужно связывать с `teamfights.csv`, где указаны `start`, `end` и `last_death`.

In [29]:
teamfights_count = (
    teamfights
    .groupby("match_id")
    .size()
    .reset_index(name="teamfights_count")
)

teamfights_players_count = (
    teamfights_players
    .groupby("match_id")
    .size()
    .reset_index(name="teamfights_players_rows")
)

teamfights_relation_check = teamfights_count.merge(
    teamfights_players_count,
    on="match_id",
    how="left"
)

teamfights_relation_check["expected_players_rows"] = (
    teamfights_relation_check["teamfights_count"] * 10
)

teamfights_relation_check["is_correct"] = (
    teamfights_relation_check["teamfights_players_rows"]
    == teamfights_relation_check["expected_players_rows"]
)

display(teamfights_relation_check.head(10))

print("Всего матчей с командными драками:", teamfights_relation_check.shape[0])
print("Корректных соответствий:", teamfights_relation_check["is_correct"].sum())
print("Некорректных соответствий:", (~teamfights_relation_check["is_correct"]).sum())

,match_id,teamfights_count,teamfights_players_rows,expected_players_rows,is_correct
0,0,12,120,120,True
1,1,15,150,150,True
2,2,11,110,110,True
3,3,16,160,160,True
4,4,6,60,60,True
5,5,13,130,130,True
6,6,10,100,100,True
7,7,12,120,120,True
8,8,10,100,100,True
9,9,13,130,130,True


Всего матчей с командными драками: 49931
Корректных соответствий: 49931
Некорректных соответствий: 0


Проверка показала, что для каждого матча количество строк в `teamfights_players.csv` ровно в 10 раз больше количества строк в `teamfights.csv`.

Это означает, что каждой командной драке соответствует по 10 строк игроков. Поэтому таблицы можно связать через номер драки внутри матча. Такой номер будет создан по порядку строк внутри каждого `match_id`.

После объединения можно будет агрегировать вклад игроков по сторонам Radiant и Dire и использовать время драки из `teamfights.csv`.

<a id="purchase-log-table"></a>

### 3.5. Исследование таблицы `purchase_log.csv`

Таблица `purchase_log.csv` содержит события покупки предметов игроками.

Основные поля:

- `match_id` — идентификатор матча;
- `player_slot` — слот игрока;
- `item_id` — идентификатор купленного предмета;
- `time` — время покупки в секундах.

Эта таблица может быть полезна для построения временных признаков, так как покупки отражают развитие экономики и силу героев по ходу матча.

In [30]:
purchase_log = pd.read_csv(DATA_DIR / "purchase_log.csv")

print("Размер purchase_log.csv:", purchase_log.shape)

print("\nТипы данных:")
display(purchase_log.dtypes)

print("\nПервые строки:")
display(purchase_log.head(10))

print("\nПропуски:")
display(purchase_log.isna().sum())

print("\nОписательная статистика:")
display(purchase_log[["item_id", "time", "player_slot", "match_id"]].describe())

Размер purchase_log.csv: (18193745, 4)

Типы данных:


item_id        int64
time           int64
player_slot    int64
match_id       int64
dtype: object


Первые строки:


,item_id,time,player_slot,match_id
0,44,-81,0,0
1,29,-63,0,0
2,43,6,0,0
3,84,182,0,0
4,46,197,0,0
5,13,203,0,0
6,44,208,0,0
7,46,425,0,0
8,20,428,0,0
9,73,428,0,0



Пропуски:


item_id        0
time           0
player_slot    0
match_id       0
dtype: int64


Описательная статистика:


,item_id,time,player_slot,match_id
count,1.819374e+07,1.819374e+07,1.819374e+07,1.819374e+07
mean,5.697155e+01,1.038303e+03,6.591196e+01,2.501832e+04
std,5.394464e+01,8.403476e+02,6.401559e+01,1.443447e+04
min,1.000000e+00,-9.000000e+01,0.000000e+00,0.000000e+00
25%,2.500000e+01,3.650000e+02,2.000000e+00,1.252900e+04
50%,4.400000e+01,9.200000e+02,4.000000e+00,2.504200e+04
75%,5.100000e+01,1.610000e+03,1.300000e+02,3.751800e+04
max,2.540000e+02,1.591400e+04,1.320000e+02,4.999900e+04


В таблице `purchase_log.csv` каждая строка соответствует покупке предмета игроком.

Отрицательные значения `time` означают покупки до начала матча. Это не ошибка данных: в Dota 2 игроки покупают стартовые предметы ещё до выхода крипов и до нулевой минуты.

Для дальнейшей обработки такие события можно либо отнести к нулевой минуте, либо учитывать как отдельный признак стартовых покупок.

Потенциальные признаки на основе `purchase_log.csv`:

- количество покупок Radiant и Dire к текущей минуте;
- количество покупок за последние несколько минут;
- покупки отдельных ключевых предметов;
- разница покупательной активности между командами.

Для интерпретации `item_id` потребуется справочник `item_ids.csv`.

In [31]:
item_ids = pd.read_csv(DATA_DIR / "item_ids.csv")

print("Размер item_ids.csv:", item_ids.shape)
display(item_ids.head(10))

popular_items = (
    purchase_log["item_id"]
    .value_counts()
    .reset_index()
)

popular_items.columns = ["item_id", "purchase_count"]

popular_items = popular_items.merge(
    item_ids,
    on="item_id",
    how="left"
)

popular_items.head(30)

Размер item_ids.csv: (189, 2)


,item_id,item_name
0,1,blink
1,2,blades_of_attack
2,3,broadsword
3,4,chainmail
4,5,claymore
5,6,helm_of_iron_will
6,7,javelin
7,8,mithril_hammer
8,9,platemail
9,10,quarterstaff


,item_id,purchase_count,item_name
0,46,3864464,tpscroll
1,42,1214710,ward_observer
2,16,745330,branches
3,29,551272,boots
4,44,490735,tango
5,43,466971,ward_sentry
6,218,453635,ward_dispenser
7,20,412626,circlet
8,38,369774,clarity
9,21,300991,ogre_axe


Чаще всего в `purchase_log.csv` встречаются расходники и базовые предметы: `tpscroll`, варды, ветки, tango, boots и другие дешёвые покупки.

Это важно для интерпретации: количество покупок само по себе не всегда означает преимущество команды. Команда может покупать много дешёвых расходников, но это не равно высокой силе героев.

Так как в текущем датасете нет стоимости предметов, таблицу `purchase_log.csv` на базовом этапе разумнее использовать осторожно:

- как счётчик покупок по сторонам;
- как счётчик покупок за последние минуты;
- как признак наличия отдельных ключевых предметов.

Основную экономическую силу команды лучше оценивать через `gold_t` из `player_time.csv`, так как этот показатель уже отражает накопленное золото игроков по времени.

<a id="ability-upgrades-table"></a>

### 3.6. Исследование таблицы `ability_upgrades.csv`

Таблица `ability_upgrades.csv` содержит события изучения способностей героями.

Основные поля:

- `match_id` — идентификатор матча;
- `player_slot` — слот игрока;
- `ability` — идентификатор способности;
- `level` — уровень героя, на котором была изучена способность;
- `time` — время изучения способности в секундах.

Эта таблица может отражать темп развития героев: чем быстрее команда получает уровни, тем раньше появляются новые способности и таланты.

In [32]:
ability_upgrades = pd.read_csv(DATA_DIR / "ability_upgrades.csv")

print("Размер ability_upgrades.csv:", ability_upgrades.shape)

print("\nТипы данных:")
display(ability_upgrades.dtypes)

print("\nПервые строки:")
display(ability_upgrades.head(10))

print("\nПропуски:")
display(ability_upgrades.isna().sum())

print("\nОписательная статистика:")
display(ability_upgrades[["ability", "level", "time", "player_slot", "match_id"]].describe())

Размер ability_upgrades.csv: (8939599, 5)

Типы данных:


ability        int64
level          int64
time           int64
player_slot    int64
match_id       int64
dtype: object


Первые строки:


,ability,level,time,player_slot,match_id
0,5448,1,326,0,0
1,5450,2,452,0,0
2,5450,3,582,0,0
3,5448,4,804,0,0
4,5450,5,916,0,0
5,5452,6,1077,0,0
6,5450,7,1222,0,0
7,5448,8,1380,0,0
8,5448,9,1554,0,0
9,5451,10,1694,0,0



Пропуски:


ability        0
level          0
time           0
player_slot    0
match_id       0
dtype: int64


Описательная статистика:


,ability,level,time,player_slot,match_id
count,8.939599e+06,8.939599e+06,8.939599e+06,8.939599e+06,8.939599e+06
mean,5.204870e+03,9.974747e+00,1.495252e+03,6.594974e+01,2.500666e+04
std,1.949415e+02,5.963747e+00,8.401830e+02,6.401578e+01,1.442761e+04
min,5.002000e+03,1.000000e+00,1.240000e+02,0.000000e+00,0.000000e+00
25%,5.025000e+03,5.000000e+00,7.610000e+02,2.000000e+00,1.253000e+04
50%,5.136000e+03,9.000000e+00,1.368000e+03,4.000000e+00,2.502100e+04
75%,5.361000e+03,1.400000e+01,2.103000e+03,1.300000e+02,3.749400e+04
max,5.654000e+03,2.500000e+01,7.063000e+03,1.320000e+02,4.999900e+04


Таблица `ability_upgrades.csv` отражает развитие героев по ходу матча.

Каждая строка соответствует изучению способности на определённом уровне героя.  
Пропусков в таблице нет, а поле `time` позволяет агрегировать события к временным срезам матча.

Для задачи прогнозирования эта таблица может быть полезна как показатель темпа развития команды:

- количество изученных способностей Radiant и Dire к текущей минуте;
- разница количества изученных способностей;
- средний достигнутый уровень героев команды;
- скорость получения новых уровней.

Максимальное значение `level` в таблице равно 25. Это показывает, что датасет относится к более старой версии Dota 2. В современных версиях игры система уровней, талантов и способностей отличается, поэтому признаки из этой таблицы нужно интерпретировать в рамках исторического состояния игры.

Конкретный `ability_id` сам по себе сложно интерпретировать без связки с героем, поэтому на базовом этапе разумнее использовать агрегаты по количеству и уровню прокачек, а не отдельные способности.

In [33]:
ability_ids = pd.read_csv(DATA_DIR / "ability_ids.csv")

print("Размер ability_ids.csv:", ability_ids.shape)

print("\nПервые строки справочника способностей:")
display(ability_ids.head(10))

popular_abilities = (
    ability_upgrades["ability"]
    .value_counts()
    .reset_index()
)

popular_abilities.columns = ["ability_id", "upgrade_count"]

popular_abilities = popular_abilities.merge(
    ability_ids,
    on="ability_id",
    how="left"
)

print("\nСамые часто изучаемые способности:")
display(popular_abilities.head(30))

print("\nКоличество способностей без расшифровки:")
print(popular_abilities["ability_name"].isna().sum())

Размер ability_ids.csv: (688, 2)

Первые строки справочника способностей:


,ability_id,ability_name
0,0,ability_base
1,5001,default_attack
2,5002,attribute_bonus
3,5003,antimage_mana_break
4,5004,antimage_blink
5,5005,antimage_spell_shield
6,5006,antimage_mana_void
7,5007,axe_berserkers_call
8,5008,axe_battle_hunger
9,5009,axe_counter_helix



Самые часто изучаемые способности:


,ability_id,upgrade_count,ability_name
0,5002,1832017,attribute_bonus
1,5131,82870,windrunner_powershot
2,5130,82565,windrunner_shackleshot
3,5132,79417,windrunner_windrun
4,5062,67865,nevermore_necromastery
5,5063,65590,nevermore_dark_lord
6,5372,63673,invoker_exort
7,5371,62738,invoker_wex
8,5133,57383,windrunner_focusfire
9,5370,53947,invoker_quas



Количество способностей без расшифровки:
0


Справочник `ability_ids.csv` полностью покрывает способности из `ability_upgrades.csv`: среди изученных способностей нет идентификаторов без расшифровки.

Самой частой записью является `attribute_bonus`. Это связано с исторической версией Dota 2: раньше вместо современной системы талантов герои могли изучать бонус к атрибутам.

Несмотря на наличие расшифровки, отдельные способности зависят от героя и игровой версии. Поэтому для базового временного датасета разумнее использовать агрегированные признаки прокачки: количество изученных способностей, максимальный уровень и темп получения уровней по сторонам Radiant и Dire.

<a id="players-table"></a>

### 3.7. Исследование таблицы `players.csv`

Таблица `players.csv` содержит итоговую информацию об игроках в каждом матче.

В ней есть данные о герое, слоте игрока, предметах, итоговом золоте, опыте, убийствах, смертях, уроне, лечении, действиях игрока и других агрегированных показателях.

Для задачи прогнозирования вероятности победы по временным срезам эту таблицу нужно анализировать осторожно: многие признаки в ней известны только после завершения матча и не могут использоваться как признаки состояния игры на конкретной минуте.

In [34]:
players = pd.read_csv(DATA_DIR / "players.csv")

print("Размер players.csv:", players.shape)

print("\nТипы данных:")
display(players.dtypes)

print("\nПервые строки:")
display(players.head(10))

print("\nПропуски:")
display(players.isna().sum().sort_values(ascending=False).head(20))

Размер players.csv: (500000, 73)

Типы данных:


match_id                               int64
account_id                             int64
hero_id                                int64
player_slot                            int64
gold                                   int64
gold_spent                             int64
gold_per_min                           int64
xp_per_min                             int64
kills                                  int64
deaths                                 int64
assists                                int64
denies                                 int64
last_hits                              int64
stuns                                float64
hero_damage                            int64
hero_healing                           int64
tower_damage                           int64
item_0                                 int64
item_1                                 int64
item_2                                 int64
item_3                                 int64
item_4                                 int64
item_5    


Первые строки:


,match_id,account_id,hero_id,player_slot,gold,gold_spent,gold_per_min,xp_per_min,kills,deaths,assists,denies,last_hits,stuns,hero_damage,hero_healing,tower_damage,item_0,item_1,item_2,item_3,item_4,item_5,level,leaver_status,xp_hero,xp_creep,xp_roshan,xp_other,gold_other,gold_death,gold_buyback,gold_abandon,gold_sell,gold_destroying_structure,gold_killing_heros,gold_killing_creeps,gold_killing_roshan,gold_killing_couriers,unit_order_none,unit_order_move_to_position,unit_order_move_to_target,unit_order_attack_move,unit_order_attack_target,unit_order_cast_position,unit_order_cast_target,unit_order_cast_target_tree,unit_order_cast_no_target,unit_order_cast_toggle,unit_order_hold_position,unit_order_train_ability,unit_order_drop_item,unit_order_give_item,unit_order_pickup_item,unit_order_pickup_rune,unit_order_purchase_item,unit_order_sell_item,unit_order_disassemble_item,unit_order_move_item,unit_order_cast_toggle_auto,unit_order_stop,unit_order_taunt,unit_order_buyback,unit_order_glyph,unit_order_eject_item_from_stash,unit_order_cast_rune,unit_order_ping_ability,unit_order_move_to_direction,unit_order_patrol,unit_order_vector_target_position,unit_order_radar,unit_order_set_item_combine_lock,unit_order_continue
0,0,0,86,0,3261,10960,347,362,9,3,18,1,30,76.7356,8690,218,143,180,37,73,56,108,0,16,0,8840.0,5440.0,NaN,83.0,50.0,-957.0,NaN,NaN,212.0,3120.0,5145.0,1087.0,400.0,NaN,NaN,4070.0,1.0,25.0,416.0,51.0,144.0,3.0,71.0,NaN,188.0,16.0,NaN,NaN,NaN,2.0,35.0,2.0,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1,51,1,2954,17760,494,659,13,3,18,9,109,87.4164,23747,0,423,46,63,119,102,24,108,22,0,14331.0,8440.0,2683.0,671.0,395.0,-1137.0,NaN,NaN,1650.0,3299.0,6676.0,4317.0,937.0,NaN,NaN,5894.0,214.0,165.0,1031.0,98.0,39.0,4.0,439.0,NaN,346.0,22.0,NaN,NaN,12.0,52.0,30.0,4.0,NaN,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN
2,0,0,83,2,110,12195,350,385,0,4,15,1,58,NaN,4217,1595,399,48,60,59,108,65,0,17,0,6692.0,8112.0,NaN,453.0,259.0,-1436.0,-1015.0,NaN,NaN,3142.0,2418.0,3697.0,400.0,NaN,NaN,7053.0,3.0,132.0,645.0,36.0,160.0,20.0,373.0,NaN,643.0,17.0,5.0,NaN,7.0,8.0,28.0,NaN,1.0,18.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN
3,0,2,11,3,1179,22505,599,605,8,4,19,6,271,NaN,14832,2714,6055,63,147,154,164,79,160,21,0,8583.0,14230.0,894.0,293.0,100.0,-2156.0,NaN,NaN,938.0,4714.0,4104.0,10432.0,400.0,NaN,NaN,4712.0,133.0,163.0,690.0,9.0,15.0,7.0,406.0,NaN,150.0,21.0,NaN,NaN,1.0,9.0,45.0,7.0,NaN,14.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN
4,0,3,67,4,3307,23825,613,762,20,3,17,13,245,NaN,33740,243,1833,114,92,147,0,137,63,24,0,15814.0,14325.0,NaN,62.0,NaN,-1437.0,-1056.0,NaN,4194.0,3217.0,7467.0,9220.0,400.0,NaN,NaN,3853.0,7.0,7.0,1173.0,31.0,84.0,8.0,198.0,NaN,111.0,23.0,1.0,NaN,NaN,2.0,44.0,6.0,NaN,13.0,NaN,NaN,NaN,1.0,3.0,NaN,NaN,23.0,NaN,NaN,NaN,NaN,NaN,NaN
5,0,4,106,128,476,12285,397,524,5,6,8,5,162,NaN,10725,0,112,145,73,149,48,212,0,19,0,8502.0,12259.0,NaN,1.0,NaN,-2394.0,-2240.0,NaN,200.0,320.0,5281.0,6193.0,NaN,NaN,NaN,6593.0,166.0,76.0,832.0,196.0,3.0,5.0,96.0,2.0,161.0,19.0,NaN,NaN,2.0,NaN,36.0,3.0,NaN,3.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN
6,0,0,102,129,317,10355,303,369,4,13,5,2,107,NaN,15028,764,0,50,11,102,36,185,81,16,0,5201.0,9417.0,NaN,1.0,NaN,-3287.0,NaN,NaN,262.0,320.0,3396.0,4356.0,NaN,NaN,NaN,3325.0,63.0,100.0,609.0,13.0,173.0,14.0,168.0,NaN,118.0,16.0,NaN,NaN,1.0,1.0,43.0,3.0,NaN,13.0,NaN,NaN,NaN,NaN,4.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
7,0,5,46,130,2390,13395,452,517,4,8,6,31,208,NaN,10230,0,2438,41,63,36,147,168,21,19,0,6853.0,13396.0,NaN,244.0,107.0,-3682.0,NaN,NaN,242.0,695.0,4350.0,8797.0,NaN,NaN,NaN,13557.0,11.0,214.0,3386.0,122.0,NaN,3.0,506.0,NaN,491.0,18.0,2.0,3.0,18.0,18.0,30.0,1.0,NaN,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,110.0,NaN,NaN,NaN,NaN,NaN
8,0,0,7,131,475,5035,189,223,1,14,8,0,27,67.0277,4774,0,0,36,0,0,46,0,180,12,0,4798.0,4038.0,NaN,27.0,NaN,-3286.0,-39.0,NaN,NaN,320.0,2127.0,1089.0,NaN,NaN,NaN,2217.0,55.0,5.0,


Пропуски:


unit_order_vector_target_position    500000
unit_order_radar                     500000
unit_order_taunt                     500000
unit_order_continue                  500000
unit_order_set_item_combine_lock     500000
unit_order_patrol                    500000
unit_order_none                      499994
unit_order_cast_rune                 499991
unit_order_move_to_direction         496449
unit_order_disassemble_item          485446
gold_abandon                         479366
unit_order_eject_item_from_stash     468736
unit_order_cast_toggle_auto          463670
unit_order_stop                      412425
gold_killing_couriers                403021
unit_order_cast_toggle               401211
unit_order_give_item                 394631
gold_buyback                         352859
unit_order_buyback                   352233
xp_roshan                            320438
dtype: int64

`players.csv` содержит одну строку на игрока в матче.  
Так как в каждом матче участвует 10 игроков, для 50 000 матчей таблица содержит 500 000 строк.

Для задачи прогнозирования вероятности победы по ходу матча эту таблицу нужно использовать осторожно. Большинство признаков являются итоговыми и известны только после завершения матча.

К таким признакам относятся:

- итоговые `kills`, `deaths`, `assists`;
- итоговые `gold`, `gold_spent`, `gold_per_min`, `xp_per_min`;
- итоговые `last_hits`, `denies`, `hero_damage`, `tower_damage`;
- финальные предметы `item_0–item_5`;
- итоговый `level`;
- агрегированные `unit_order_*` признаки.

Использование этих признаков для прогноза на конкретной минуте привело бы к утечке данных из будущего.

Полезными без утечки могут быть признаки, известные до начала или в начале матча:

- `match_id`;
- `player_slot`;
- `hero_id`;
- состав героев Radiant и Dire.

Поэтому `players.csv` будет использоваться преимущественно для анализа состава команд и связи игроков со сторонами, но не как источник итоговой игровой статистики для временного прогноза.

In [35]:
players_slot_summary = (
    players["player_slot"]
    .value_counts()
    .sort_index()
    .reset_index()
)

players_slot_summary.columns = ["player_slot", "rows_count"]

players_slot_summary["side"] = np.where(
    players_slot_summary["player_slot"].between(0, 4),
    "Radiant",
    "Dire"
)

players_slot_summary

,player_slot,rows_count,side
0,0,50000,Radiant
1,1,50000,Radiant
2,2,50000,Radiant
3,3,50000,Radiant
4,4,50000,Radiant
5,128,50000,Dire
6,129,50000,Dire
7,130,50000,Dire
8,131,50000,Dire
9,132,50000,Dire


Распределение `player_slot` подтверждает корректную структуру таблицы `players.csv`.

Каждый из 10 игровых слотов встречается ровно 50 000 раз, то есть каждый матч представлен полным составом из 10 игроков.

Для дальнейшей обработки можно использовать следующую интерпретацию:

- `0–4` — игроки Radiant;
- `128–132` — игроки Dire.

Это соответствие будет использоваться при агрегации признаков по командам.

In [36]:
hero_names = pd.read_csv(DATA_DIR / "hero_names.csv")

print("Размер hero_names.csv:", hero_names.shape)

print("\nПервые строки справочника героев:")
display(hero_names.head(10))

hero_pick_counts = (
    players["hero_id"]
    .value_counts()
    .reset_index()
)

hero_pick_counts.columns = ["hero_id", "pick_count"]

hero_pick_counts = hero_pick_counts.merge(
    hero_names[["hero_id", "localized_name"]],
    on="hero_id",
    how="left"
)

print("\nСамые часто встречающиеся герои:")
display(hero_pick_counts.head(30))

print("\nКоличество hero_id без расшифровки:")
print(hero_pick_counts["localized_name"].isna().sum())

Размер hero_names.csv: (112, 3)

Первые строки справочника героев:


,name,hero_id,localized_name
0,npc_dota_hero_antimage,1,Anti-Mage
1,npc_dota_hero_axe,2,Axe
2,npc_dota_hero_bane,3,Bane
3,npc_dota_hero_bloodseeker,4,Bloodseeker
4,npc_dota_hero_crystal_maiden,5,Crystal Maiden
5,npc_dota_hero_drow_ranger,6,Drow Ranger
6,npc_dota_hero_earthshaker,7,Earthshaker
7,npc_dota_hero_juggernaut,8,Juggernaut
8,npc_dota_hero_mirana,9,Mirana
9,npc_dota_hero_morphling,10,Morphling



Самые часто встречающиеся герои:


,hero_id,pick_count,localized_name
0,21,20881,Windranger
1,11,17007,Shadow Fiend
2,74,11676,Invoker
3,7,11323,Earthshaker
4,28,11181,Slardar
5,39,10590,Queen of Pain
6,8,10394,Juggernaut
7,100,10306,Tusk
8,73,9823,Alchemist
9,14,9447,Pudge



Количество hero_id без расшифровки:
1


Справочник `hero_names.csv` позволяет расшифровать почти все `hero_id` из таблицы `players.csv`.

Самыми частыми героями в датасете являются Windranger, Shadow Fiend, Invoker, Earthshaker и Slardar. Это отражает мету и популярность героев на момент сбора данных.

Один `hero_id` не был найден в справочнике. Такой случай нужно проверить отдельно: это может быть устаревший, служебный или отсутствующий в справочнике идентификатор.

In [37]:
missing_hero_ids = (
    hero_pick_counts[hero_pick_counts["localized_name"].isna()]
    .sort_values("pick_count", ascending=False)
)

missing_hero_ids

,hero_id,pick_count,localized_name
110,0,37,NaN


В `players.csv` найдено 37 строк с `hero_id = 0`, для которых нет расшифровки в `hero_names.csv`.  
Так как таких строк крайне мало, их можно рассматривать как некорректные или служебные записи. При построении признаков по героям такие строки можно исключить или оставить как отдельную категорию `unknown_hero`.

<a id="match-table"></a>

### 3.8. Исследование таблицы `match.csv`

Таблица `match.csv` содержит одну строку на матч и включает общую информацию о матче.

Для текущей задачи особенно важны:

- `match_id` — идентификатор матча;
- `start_time` — время начала матча;
- `duration` — длительность матча;
- `game_mode` — режим игры;
- `radiant_win` — целевой признак;
- `cluster` — серверный кластер.

Некоторые поля, например `tower_status_radiant`, `tower_status_dire`, `barracks_status_radiant` и `barracks_status_dire`, отражают состояние зданий в конце матча. Для прогноза вероятности победы по ходу игры их нельзя использовать как признаки, так как они содержат информацию из будущего.

In [38]:
matches = pd.read_csv(DATA_DIR / "match.csv")

print("Размер match.csv:", matches.shape)

print("\nТипы данных:")
display(matches.dtypes)

print("\nПервые строки:")
display(matches.head(10))

print("\nПропуски:")
display(matches.isna().sum())

print("\nОписательная статистика:")
display(matches.describe())

Размер match.csv: (50000, 13)

Типы данных:


match_id                   int64
start_time                 int64
duration                   int64
tower_status_radiant       int64
tower_status_dire          int64
barracks_status_dire       int64
barracks_status_radiant    int64
first_blood_time           int64
game_mode                  int64
radiant_win                 bool
negative_votes             int64
positive_votes             int64
cluster                    int64
dtype: object


Первые строки:


,match_id,start_time,duration,tower_status_radiant,tower_status_dire,barracks_status_dire,barracks_status_radiant,first_blood_time,game_mode,radiant_win,negative_votes,positive_votes,cluster
0,0,1446750112,2375,1982,4,3,63,1,22,True,0,1,155
1,1,1446753078,2582,0,1846,63,0,221,22,False,0,2,154
2,2,1446764586,2716,256,1972,63,48,190,22,False,0,0,132
3,3,1446765723,3085,4,1924,51,3,40,22,False,0,0,191
4,4,1446796385,1887,2047,0,0,63,58,22,True,0,0,156
5,5,1446798766,1574,2047,4,3,63,113,22,True,0,0,155
6,6,1446800938,2124,1972,0,3,63,4,22,True,0,0,151
7,7,1446804030,2328,2046,0,0,63,255,22,True,0,0,138
8,8,1446819063,2002,0,1982,63,0,4,22,False,0,0,182
9,9,1446837251,2961,0,1972,63,0,85,22,False,0,0,133



Пропуски:


match_id                   0
start_time                 0
duration                   0
tower_status_radiant       0
tower_status_dire          0
barracks_status_dire       0
barracks_status_radiant    0
first_blood_time           0
game_mode                  0
radiant_win                0
negative_votes             0
positive_votes             0
cluster                    0
dtype: int64


Описательная статистика:


,match_id,start_time,duration,tower_status_radiant,tower_status_dire,barracks_status_dire,barracks_status_radiant,first_blood_time,game_mode,negative_votes,positive_votes,cluster
count,50000.000000,5.000000e+04,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,24999.500000,1.447573e+09,2476.453500,1000.016440,935.250060,34.529460,34.77526,93.825520,21.468000,0.015480,0.036820,142.304720
std,14433.901067,1.485270e+05,634.631261,948.211846,937.974714,29.209672,29.73214,92.648332,3.218258,0.364696,0.871068,25.156608
min,0.000000,1.446750e+09,59.000000,0.000000,0.000000,0.000000,0.00000,0.000000,2.000000,0.000000,0.000000,111.000000
25%,12499.750000,1.447456e+09,2029.000000,0.000000,0.000000,0.000000,0.00000,9.000000,22.000000,0.000000,0.000000,123.000000
50%,24999.500000,1.447577e+09,2415.000000,1536.000000,384.000000,51.000000,51.00000,77.000000,22.000000,0.000000,0.000000,133.000000
75%,37499.250000,1.447700e+09,2872.000000,1974.000000,1972.000000,63.000000,63.00000,144.000000,22.000000,0.000000,0.000000,154.000000
max,49999.000000,1.447829e+09,16037.000000,2047.000000,2047.000000,63.000000,63.00000,831.000000,22.000000,47.000000,80.000000,242.000000


В `match.csv` каждая строка соответствует одному матчу.  
Пропусков в таблице нет, а поле `radiant_win` можно использовать как целевой признак.

Часть признаков таблицы описывает итоговое состояние матча. Например, `tower_status_radiant`, `tower_status_dire`, `barracks_status_radiant` и `barracks_status_dire` отражают состояние зданий после завершения игры. Для прогноза вероятности победы по ходу матча такие признаки использовать нельзя, так как они создают утечку данных из будущего.

Поле `duration` также известно только после завершения матча, поэтому его можно использовать для анализа длительности, но не как признак модели для live-прогноза.

Минимальная и максимальная длительность матчей указывают на наличие аномально коротких и очень длинных игр, которые нужно дополнительно изучить.

In [39]:
matches_analysis = matches.copy()
matches_analysis["start_datetime"] = pd.to_datetime(
    matches_analysis["start_time"],
    unit="s"
)

matches_analysis["duration_min"] = matches_analysis["duration"] / 60

print("Распределение целевой переменной:")
display(
    matches_analysis["radiant_win"]
    .value_counts()
    .rename(index={True: "Radiant win", False: "Dire win"})
    .reset_index()
    .rename(columns={"index": "result", "radiant_win": "matches_count"})
)

print("\nДоля классов:")
display(
    matches_analysis["radiant_win"]
    .value_counts(normalize=True)
    .rename(index={True: "Radiant win", False: "Dire win"})
    .reset_index()
    .rename(columns={"index": "result", "radiant_win": "share"})
)

print("\nРаспределение game_mode:")
display(matches_analysis["game_mode"].value_counts().reset_index())

print("\nПериод матчей:")
print(matches_analysis["start_datetime"].min())
print(matches_analysis["start_datetime"].max())

print("\nДлительность матчей в минутах:")
display(matches_analysis["duration_min"].describe())

Распределение целевой переменной:


,matches_count,count
0,Radiant win,25943
1,Dire win,24057



Доля классов:


,share,proportion
0,Radiant win,0.51886
1,Dire win,0.48114



Распределение game_mode:


,game_mode,count
0,22,48670
1,2,1330



Период матчей:
2015-11-05 19:01:52
2015-11-18 06:46:55

Длительность матчей в минутах:


count    50000.000000
mean        41.274225
std         10.577188
min          0.983333
25%         33.816667
50%         40.250000
75%         47.866667
max        267.283333
Name: duration_min, dtype: float64

Целевая переменная `radiant_win` почти сбалансирована: Radiant выиграли около 51.9% матчей, Dire — около 48.1%. Поэтому сильного дисбаланса классов нет.

Большинство матчей относятся к режиму `game_mode = 22`. Также присутствует небольшая группа матчей с `game_mode = 2`, которую нужно учитывать при дальнейшем анализе.

Датасет охватывает короткий исторический период: с 5 по 18 ноября 2015 года. Это подтверждает, что данные относятся к старой версии Dota 2 и не отражают текущее состояние игры.

Средняя длительность матча составляет около 41 минуты. При этом в данных есть аномально короткие и аномально длинные матчи. Такие игры могут искажать обучение модели, поэтому перед построением итогового датасета их нужно дополнительно проверить.

In [40]:
short_matches = matches_analysis[matches_analysis["duration_min"] < 10]
long_matches = matches_analysis[matches_analysis["duration_min"] > 90]

print("Матчи короче 10 минут:", short_matches.shape[0])
print("Матчи длиннее 90 минут:", long_matches.shape[0])

print("\nКороткие матчи:")
display(
    short_matches[
        ["match_id", "start_datetime", "duration_min", "radiant_win", "game_mode"]
    ].sort_values("duration_min").head(20)
)

print("\nДлинные матчи:")
display(
    long_matches[
        ["match_id", "start_datetime", "duration_min", "radiant_win", "game_mode"]
    ].sort_values("duration_min", ascending=False).head(20)
)

Матчи короче 10 минут: 52
Матчи длиннее 90 минут: 12

Короткие матчи:


,match_id,start_datetime,duration_min,radiant_win,game_mode
31438,31438,2015-11-15 22:10:50,0.983333,False,22
13914,13914,2015-11-14 03:02:04,1.050000,False,22
21228,21228,2015-11-14 22:38:18,2.450000,False,22
6213,6213,2015-11-13 00:41:55,2.466667,True,22
24474,24474,2015-11-15 07:05:32,3.366667,False,22
31800,31800,2015-11-15 22:54:54,3.566667,False,22
40993,40993,2015-11-17 03:44:40,3.616667,True,22
24220,24220,2015-11-15 06:22:23,4.400000,False,22
1221,1221,2015-11-12 11:23:25,4.533333,False,22
44908,44908,2015-11-17 17:15:47,4.566667,True,22



Длинные матчи:


,match_id,start_datetime,duration_min,radiant_win,game_mode
9946,9946,2015-11-13 16:34:59,267.283333,False,22
27373,27373,2015-11-15 14:47:53,111.216667,True,22
26680,26680,2015-11-15 13:21:27,110.366667,False,22
22046,22046,2015-11-15 00:26:07,109.733333,True,22
1610,1610,2015-11-12 13:05:56,100.633333,False,2
10159,10159,2015-11-13 17:19:30,98.833333,True,22
30548,30548,2015-11-15 20:36:12,94.950000,True,22
28445,28445,2015-11-15 16:52:37,94.083333,True,22
39388,39388,2015-11-16 23:14:31,93.666667,False,22
12515,12515,2015-11-13 23:13:34,93.216667,True,22


В датасете обнаружены редкие матчи с аномальной длительностью: 52 матча короче 10 минут и 12 матчей длиннее 90 минут.

Такие наблюдения составляют малую долю от общего объёма данных, но могут искажать построение временного ряда. Очень короткие матчи часто не отражают нормальный игровой процесс, а очень длинные создают редкие временные участки, плохо представленные в остальных данных.

При формировании итогового датасета для моделирования такие матчи целесообразно исключить, ограничив длительность матчей диапазоном от 10 до 90 минут.

<a id="player-time-table"></a>

### 3.9. Исследование таблицы `player_time.csv`

Таблица `player_time.csv` является основным источником временной структуры.

В ней каждая строка соответствует временному срезу конкретного матча.  
Поле `times` задаёт время среза, а признаки вида `gold_t_*`, `xp_t_*`, `lh_t_*` содержат значения золота, опыта и добиваний для игроков разных слотов.

In [41]:
player_time = pd.read_csv(DATA_DIR / "player_time.csv")

print("Размер player_time.csv:", player_time.shape)

print("\nТипы данных:")
display(player_time.dtypes)

print("\nПервые строки:")
display(player_time.head(10))

print("\nПропуски:")
display(player_time.isna().sum())

print("\nОписательная статистика по times:")
display(player_time["times"].describe())

Размер player_time.csv: (2209778, 32)

Типы данных:


match_id      int64
times         int64
gold_t_0      int64
lh_t_0        int64
xp_t_0        int64
gold_t_1      int64
lh_t_1        int64
xp_t_1        int64
gold_t_2      int64
lh_t_2        int64
xp_t_2        int64
gold_t_3      int64
lh_t_3        int64
xp_t_3        int64
gold_t_4      int64
lh_t_4        int64
xp_t_4        int64
gold_t_128    int64
lh_t_128      int64
xp_t_128      int64
gold_t_129    int64
lh_t_129      int64
xp_t_129      int64
gold_t_130    int64
lh_t_130      int64
xp_t_130      int64
gold_t_131    int64
lh_t_131      int64
xp_t_131      int64
gold_t_132    int64
lh_t_132      int64
xp_t_132      int64
dtype: object


Первые строки:


,match_id,times,gold_t_0,lh_t_0,xp_t_0,gold_t_1,lh_t_1,xp_t_1,gold_t_2,lh_t_2,xp_t_2,gold_t_3,lh_t_3,xp_t_3,gold_t_4,lh_t_4,xp_t_4,gold_t_128,lh_t_128,xp_t_128,gold_t_129,lh_t_129,xp_t_129,gold_t_130,lh_t_130,xp_t_130,gold_t_131,lh_t_131,xp_t_131,gold_t_132,lh_t_132,xp_t_132
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,60,409,0,63,142,1,186,168,0,125,200,0,193,194,1,125,174,2,77,138,1,62,345,6,351,100,0,77,613,1,125
2,0,120,546,0,283,622,4,645,330,0,376,345,1,698,628,5,374,354,4,437,673,5,543,684,12,805,200,0,210,815,5,323
3,0,180,683,1,314,927,9,1202,430,0,376,644,6,1172,806,7,570,614,8,829,895,8,842,958,16,1135,300,0,210,1290,8,527
4,0,240,956,1,485,1264,11,1583,530,0,391,919,11,1610,1281,10,1216,1082,8,1318,1087,10,1048,1500,26,1842,400,0,210,1431,9,589
5,0,300,1056,1,649,1451,13,1810,630,0,504,1102,15,1888,1708,17,1633,1300,11,1901,1233,11,1352,1841,32,2162,500,0,241,2110,17,918
6,0,360,1156,1,680,1744,18,2192,730,0,535,1326,18,2197,2339,21,2203,1520,14,2231,1497,15,1806,2186,38,2554,600,0,241,2623,28,1248
7,0,420,1257,2,778,2053,23,2584,830,0,566,1711,25,2558,2693,27,2574,1620,14,2231,1761,19,2033,2652,47,3111,700,0,241,3033,28,1329
8,0,480,1809,3,1135,2536,32,3153,1328,0,1144,2075,32,3161,3269,31,3046,1720,14,2384,2126,25,2575,3196,57,3715,858,1,300,3380,32,1535
9,0,540,2111,3,1393,3033,33,3449,1510,1,1306,2386,39,3398,3606,36,3361,2194,22,2905,2306,27,2967,3629,65,4169,958,1,300,3648,34,1697



Пропуски:


match_id      0
times         0
gold_t_0      0
lh_t_0        0
xp_t_0        0
gold_t_1      0
lh_t_1        0
xp_t_1        0
gold_t_2      0
lh_t_2        0
xp_t_2        0
gold_t_3      0
lh_t_3        0
xp_t_3        0
gold_t_4      0
lh_t_4        0
xp_t_4        0
gold_t_128    0
lh_t_128      0
xp_t_128      0
gold_t_129    0
lh_t_129      0
xp_t_129      0
gold_t_130    0
lh_t_130      0
xp_t_130      0
gold_t_131    0
lh_t_131      0
xp_t_131      0
gold_t_132    0
lh_t_132      0
xp_t_132      0
dtype: int64


Описательная статистика по times:


count    2.209778e+06
mean     1.371875e+03
std      8.965257e+02
min      0.000000e+00
25%      6.600000e+02
50%      1.320000e+03
75%      1.980000e+03
max      1.614000e+04
Name: times, dtype: float64

`player_time.csv` является основной таблицей для построения временного ряда.

Каждая строка соответствует временному срезу матча.  
Поле `times` хранит время в секундах, при этом значения идут с шагом 60 секунд, то есть данные представлены поминутно.

Для каждого игрока доступны три показателя:

- `gold_t_*` — накопленное золото;
- `xp_t_*` — накопленный опыт;
- `lh_t_*` — количество добиваний крипов.

Слоты `0–4` соответствуют игрокам Radiant, а слоты `128–132` — игрокам Dire.  
На основе этих признаков можно агрегировать командные показатели и построить признаки преимущества Radiant над Dire.

In [42]:
time_steps_by_match = (
    player_time
    .groupby("match_id")["times"]
    .agg(["count", "min", "max"])
    .reset_index()
)

time_steps_by_match["max_minute"] = time_steps_by_match["max"] / 60

print("Количество матчей в player_time.csv:", time_steps_by_match.shape[0])

print("\nКоличество временных срезов на матч:")
display(time_steps_by_match["count"].describe())

print("\nМаксимальная минута матча по player_time:")
display(time_steps_by_match["max_minute"].describe())

time_diffs = (
    player_time
    .sort_values(["match_id", "times"])
    .groupby("match_id")["times"]
    .diff()
    .dropna()
)

print("\nУникальные шаги между временными срезами:")
display(time_diffs.value_counts().sort_index())

Количество матчей в player_time.csv: 50000

Количество временных срезов на матч:


count    50000.000000
mean        44.195560
std         10.581881
min          4.000000
25%         37.000000
50%         43.000000
75%         51.000000
max        270.000000
Name: count, dtype: float64


Максимальная минута матча по player_time:


count    50000.000000
mean        43.195560
std         10.581881
min          3.000000
25%         36.000000
50%         42.000000
75%         50.000000
max        269.000000
Name: max_minute, dtype: float64


Уникальные шаги между временными срезами:


times
60.0    2159778
Name: count, dtype: int64

В таблице `player_time.csv` представлены все 50 000 матчей.

Проверка временных шагов показала, что все соседние срезы внутри матча отличаются ровно на 60 секунд. Значит временной ряд является регулярным и поминутным.

Количество временных срезов на матч зависит от длительности игры. Минимальное количество срезов равно 4, что соответствует аномально коротким матчам, а максимальное равно 270, что соответствует очень длинному матчу.

Эти результаты подтверждают, что `player_time.csv` можно использовать как основу для построения эпизодического временного ряда: один матч — один эпизод, одна строка — один временной срез.

In [43]:
time_duration_check = time_steps_by_match.merge(
    matches_analysis[["match_id", "duration", "duration_min"]],
    on="match_id",
    how="left"
)

time_duration_check["max_time_sec"] = time_duration_check["max"]
time_duration_check["time_to_duration_diff_sec"] = (
    time_duration_check["duration"] - time_duration_check["max_time_sec"]
)

print("Разница между duration и последним временным срезом, секунд:")
display(time_duration_check["time_to_duration_diff_sec"].describe())

print("\nМатчи с отрицательной разницей:")
display(
    time_duration_check[time_duration_check["time_to_duration_diff_sec"] < 0]
    .sort_values("time_to_duration_diff_sec")
    .head(20)
)

print("\nМатчи с большой положительной разницей больше 120 секунд:")
display(
    time_duration_check[time_duration_check["time_to_duration_diff_sec"] > 120]
    .sort_values("time_to_duration_diff_sec", ascending=False)
    .head(20)
)

Разница между duration и последним временным срезом, секунд:


count    50000.000000
mean      -115.280100
std         17.321538
min       -146.000000
25%       -130.000000
50%       -116.000000
75%       -100.000000
max        -85.000000
Name: time_to_duration_diff_sec, dtype: float64


Матчи с отрицательной разницей:


,match_id,count,min,max,max_minute,duration,duration_min,max_time_sec,time_to_duration_diff_sec
16410,16410,52,0,3060,51.0,2914,48.566667,3060,-146
23852,23852,51,0,3000,50.0,2854,47.566667,3000,-146
2765,2765,45,0,2640,44.0,2494,41.566667,2640,-146
13520,13520,41,0,2400,40.0,2254,37.566667,2400,-146
17618,17618,49,0,2880,48.0,2734,45.566667,2880,-146
20490,20490,63,0,3720,62.0,3574,59.566667,3720,-146
46568,46568,39,0,2280,38.0,2134,35.566667,2280,-146
3613,3613,51,0,3000,50.0,2854,47.566667,3000,-146
3605,3605,63,0,3720,62.0,3574,59.566667,3720,-146
45718,45718,45,0,2640,44.0,2494,41.566667,2640,-146



Матчи с большой положительной разницей больше 120 секунд:


,match_id,count,min,max,max_minute,duration,duration_min,max_time_sec,time_to_duration_diff_sec


Последний временной срез в `player_time.csv` для всех матчей находится позже официальной длительности матча из `match.csv`.

Разница составляет примерно от 85 до 146 секунд. Это похоже на технический хвост после завершения матча, возникающий при парсинге replay-данных.

Для задачи прогнозирования вероятности победы по ходу матча такие строки использовать нельзя: после завершения игры исход уже известен. Поэтому при формировании итогового временного датасета будут оставлены только строки, где `times <= duration`.

In [44]:
player_time_with_duration = player_time.merge(
    matches_analysis[["match_id", "duration", "duration_min"]],
    on="match_id",
    how="left"
)

rows_before = player_time_with_duration.shape[0]

player_time_valid_time = player_time_with_duration[
    player_time_with_duration["times"] <= player_time_with_duration["duration"]
].copy()

rows_after = player_time_valid_time.shape[0]
rows_removed = rows_before - rows_after

print("Строк до фильтрации:", rows_before)
print("Строк после фильтрации:", rows_after)
print("Удалено строк:", rows_removed)
print("Доля удалённых строк, %:", round(rows_removed / rows_before * 100, 2))

valid_time_steps_by_match = (
    player_time_valid_time
    .groupby("match_id")["times"]
    .agg(["count", "min", "max"])
    .reset_index()
)

print("\nКоличество временных срезов после фильтрации:")
display(valid_time_steps_by_match["count"].describe())

print("\nМаксимальная минута после фильтрации:")
display((valid_time_steps_by_match["max"] / 60).describe())

Строк до фильтрации: 2209778
Строк после фильтрации: 2088916
Удалено строк: 120862
Доля удалённых строк, %: 5.47

Количество временных срезов после фильтрации:


count    50000.000000
mean        41.778320
std         10.579885
min          1.000000
25%         34.000000
50%         41.000000
75%         48.000000
max        268.000000
Name: count, dtype: float64


Максимальная минута после фильтрации:


count    50000.000000
mean        40.778320
std         10.579885
min          0.000000
25%         33.000000
50%         40.000000
75%         47.000000
max        267.000000
Name: max, dtype: float64

После фильтрации `times <= duration` было удалено 120 862 строки, то есть 5.47% временных срезов.

Эти строки соответствовали техническому хвосту после официального завершения матча. Для live-прогнозирования они некорректны, потому что в этот момент исход игры уже известен.

После фильтрации среднее количество временных срезов на матч составляет около 42. Минимальное значение равно 1, что связано с аномально короткими матчами. Такие матчи уже были обнаружены ранее и будут исключены при финальной очистке данных.

<a id="tables-research-summary"></a>

### 3.10. Итоги первичного исследования таблиц

В результате первичного исследования были изучены основные, событийные и справочные таблицы датасета.

Ключевой таблицей для построения временной структуры является `player_time.csv`. Она содержит регулярные поминутные срезы матчей и задаёт основу будущего датасета в формате `match_id` + `minute`.

Таблица `match.csv` содержит целевой признак `radiant_win`, длительность матча, режим игры и серверный кластер. При этом признаки финального состояния зданий не будут использоваться как признаки модели, так как они отражают информацию после завершения игры.

Таблица `players.csv` содержит составы команд и итоговую статистику игроков. Для прогноза по ходу матча итоговые показатели игроков использовать нельзя, но `hero_id` и `player_slot` могут быть полезны как признаки состава команд.

Событийные таблицы содержат важную информацию о ходе игры:

- `objectives.csv` — башни, бараки, Рошан, Aegis и первая кровь;
- `teamfights.csv` и `teamfights_players.csv` — командные драки и вклад игроков;
- `purchase_log.csv` — покупки предметов;
- `ability_upgrades.csv` — прокачка способностей и темп развития героев.

Эти таблицы будут использоваться для расширения признакового пространства. Все событийные признаки должны агрегироваться только по событиям, которые произошли не позже текущей минуты матча. Это необходимо для предотвращения утечки данных из будущего.

Таким образом, итоговый аналитический датасет будет строиться не только на основе `player_time.csv`, а как расширенный временной датасет, объединяющий экономику, опыт, фарм, составы команд и игровые события.

<a id="dataset-building"></a>

## 4. Формирование расширенного временного датасета

На этом этапе будет сформирован аналитический датасет для прогнозирования вероятности победы Radiant по состоянию матча на конкретной минуте.

Основой датасета является поминутная временная сетка из `player_time.csv`.  
Каждая строка будущего датасета соответствует одному временному срезу конкретного матча:

- `match_id` — идентификатор матча;
- `minute` — минута матча;
- признаки состояния игры на этой минуте;
- `radiant_win` — итоговый исход матча.

Дальше эта временная сетка будет расширяться признаками из других таблиц:

- `match.csv` — целевой признак и метаинформация о матче;
- `players.csv` — составы героев Radiant и Dire;
- `objectives.csv` — башни, бараки, Рошан, Aegis, первая кровь;
- `teamfights.csv` и `teamfights_players.csv` — командные драки и их результат;
- `purchase_log.csv` — покупки предметов;
- `ability_upgrades.csv` — темп прокачки героев.

Важно, что все событийные признаки будут агрегироваться только по событиям, которые произошли не позже текущей минуты. Это нужно, чтобы не допустить утечку информации из будущего.